In [1]:
print(123)

123


In [2]:
import requests

files = {
    'rag_helper.py': 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py',
    'ingest.py': 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py',
}

for filename, url in files.items():
    r = requests.get(url)
    r.raise_for_status()
    with open(filename, 'w') as f:
        f.write(r.text)
    print(f'Downloaded {filename}')

Downloaded rag_helper.py
Downloaded ingest.py


In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

print('Client initialized successfully')

Client initialized successfully


In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. In a terminal, start a local model with:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local Ollama server is running, use:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python, install the client:
   ```bash
   pip install ollama
   ```


In [7]:
# Direct LLM call without RAG context
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.output_text

"To determine if you can join the course, please check the following:\n\n1. **Enrollment Dates**: Ensure that the enrollment period is still open.\n2. **Prerequisites**: Confirm whether you meet any prerequisites necessary for the course.\n3. **Course Capacity**: Verify if there are available spots in the course.\n4. **Registration Process**: Follow the registration guidelines provided by the institution or platform offering the course.\n\nIf you're uncertain, reach out to the course coordinator or check the official website for more information."

In [8]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ['query'],
        'additionalProperties': False
    }
}

In [10]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

response

Response(id='resp_041c7990a081147d006a1fe14f6e0081a0839694457871e2cd', created_at=1780474191.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join the course"}', call_id='call_nNqCy7iYRhbGjRZhbxJ7tXFT', name='search', type='function_call', id='fc_041c7990a081147d006a1fe151528481a0a5b52d8e3e3dcaac', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=1.0, background=False, completed_at=1780474193.0, conversation=None, max_output_tokens=None, max_tool_cal

In [11]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join the course"}', call_id='call_nNqCy7iYRhbGjRZhbxJ7tXFT', name='search', type='function_call', id='fc_041c7990a081147d006a1fe151528481a0a5b52d8e3e3dcaac', namespace=None, status='completed')

In [12]:
import json

args = json.loads(call.arguments)
args

{'query': 'join the course'}

In [13]:
call.name

'search'

In [14]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2025.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [15]:
result_json = json.dumps(results, indent=2)

In [16]:
function_call_output = {
    'type': 'function_call_output',
    'call_id': call.call_id,
    'output': json.dumps(search(**args), indent=2),
}

In [17]:
messages.append(call)
messages.append(function_call_output)

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course"}', call_id='call_nNqCy7iYRhbGjRZhbxJ7tXFT', name='search', type='function_call', id='fc_041c7990a081147d006a1fe151528481a0a5b52d8e3e3dcaac', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_nNqCy7iYRhbGjRZhbxJ7tXFT',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "ll

In [18]:
response2 = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
)

print(response2.output_text)

Yes, you can still join the course! However, if you want to receive a certificate, make sure to submit your project while submissions are still being accepted. 

Feel free to start learning and submitting homework without needing to register officially; registration is mainly for gauging interest.


In [19]:
def calculate_openai_price(input_tokens, output_tokens):
    # gpt-4o-mini pricing (per 1M tokens)
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    return {
        'input_cost': input_cost,
        'output_cost': output_cost,
        'total_cost': input_cost + output_cost
    }


usage = response2.usage
result = calculate_openai_price(usage.input_tokens, usage.output_tokens)
print('Total Cost: $', round(result['total_cost'], 8))

Total Cost: $ 0.00012915


In [20]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        'type': 'function_call_output',
        'call_id': call.call_id,
        'output': result_json,
    }

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [22]:
def agent_loop(instructions, question, model='gpt-4o-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    last_answer = ''

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            elif item.type == 'message':
                last_answer = item.content[0].text
                print('ASSISTANT:')
                print(last_answer)

        it += 1
        if not has_function_calls:
            break

    return last_answer

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [24]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! If you're looking to earn a certificate, you need to submit your project while submissions are still being accepted. 

Just a heads up, you don’t need to wait for a confirmation email; you can start learning and submitting homework right away, even without completing a formal registration. 

Is there anything else you would like to know or explore further?


In [25]:
result

"Yes, you can still join the course! If you're looking to earn a certificate, you need to submit your project while submissions are still being accepted. \n\nJust a heads up, you don’t need to wait for a confirmation email; you can start learning and submitting homework right away, even without completing a formal registration. \n\nIs there anything else you would like to know or explore further?"

In [26]:
instructions_strict = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question_offtopic = "what's queen gambit?"

result = agent_loop(instructions_strict, question_offtopic)
print(result)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
It seems that there are no relevant entries related to "queen gambit" in the FAQ database. It might be an off-topic question not related to the course content.

Is there another area related to the course or its logistics that you would like to explore?
It seems that there are no relevant entries related to "queen gambit" in the FAQ database. It might be an off-topic question not related to the course content.

Is there another area related to the course or its logistics that you would like to explore?


In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
def search(query: str) -> dict:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [31]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions_strict,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-4o-mini')
)

In [32]:
result = runner.loop(
    prompt='How do I run Ollama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [33]:
result.cost

CostInfo(input_cost=Decimal('0.0002337'), output_cost=Decimal('0.000231'), total_cost=Decimal('0.0004647'))

In [34]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results\nand then perform more searches.\n\nThe question has to be about the course or its logistics, offtopic questions\nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the\nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Ollama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"run Ollama locally"}', call_id='call_ENHFsYeeCfje8s3ugG2UOJTZ'

In [ ]:
runner.run();